## Faster R-CNN (ResNet-50 + FPN v2): RGB vs YGB vs LGB with a **colour-neutral start**

Control experiment for the question *"does the RGB-pretrained network favour RGB?"*. Compared with the original notebook, only the **starting point** changes (data, class weights, schedule, seed and evaluation are identical):

1. **Channel-independent normalisation.** The same mean/std for channel 0, 1 and 2 (`NORM_MEAN`, `NORM_STD`) instead of torchvision's ImageNet R/G/B constants, so channel 0 is not treated as "red" when it holds Y or L.
2. **Channel-neutral stem.** The first conv layer (`conv1`) keeps the COCO filters but averages them over the input channels, so the network starts with *identical* filters for channel 0, 1 and 2 (no red-green colour-opponent filters that only make sense for RGB). It is still fully trainable.
3. **Adaptive BatchNorm.** The backbone BatchNorm statistics are re-estimated on the training images of *each* colour space before training, so no colour space inherits statistics measured on COCO-RGB.

Everything deeper than `conv1` is still COCO-pretrained, so the start is *less* tilted towards RGB, not perfectly neutral. Results go to `runs_frcnn_neutral/`, so the original runs are never overwritten. Currently set to **SIMD** (15 classes: car, truck, van, longvehicle, bus, airliner, propeller, trainer, chartered, fighter, other, stairtruck, pushbacktruck, helicopter, boat). Switch dataset in **one place** (`DATASET_NAME`).


In [1]:
%pip install -q torchmetrics pycocotools pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torchmetrics
import pycocotools
import pandas as pd

print("torchmetrics:", torchmetrics.__version__)
print("pandas:", pd.__version__)
import pycocotools
print("pycocotools: OK")

torchmetrics: 1.9.0
pandas: 3.0.2
pycocotools: OK


In [3]:
import os, json, math, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection import roi_heads as tv_roi_heads
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("Using device:", device)

torch 2.7.1+cu118 | torchvision 0.22.1+cu118
GPU: NVIDIA GeForce RTX 2060 | VRAM 6.4 GB
Using device: cuda


## Configuration


In [4]:
import os

# Change to Desktop
os.chdir('C:/Users/DELL/Desktop/c')

# Verify
print("Current working directory:", os.getcwd())
print("\nContents of Desktop:")
print(os.listdir('.'))

Current working directory: C:\Users\DELL\Desktop\c

Contents of Desktop:
['.ipynb_checkpoints', 'Codes', 'Extra', 'fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth', 'Inspect_SIMD_test.m', 'Inspect_SIMD_train.m', 'Inspect_SIMD_val.m', 'Inspect_SSL_test.m', 'Inspect_SSL_train.m', 'Inspect_SSL_val.m', 'matlab', 'pretrained', 'rgb_passthrough.m', 'rgb_to_lgb.m', 'rgb_to_ygb.m', 'rtdetr-l.pt', 'rtdetr_r18vd_dec3_6x_coco_from_paddle.pth', 'runs', 'runs_frcnn', 'runs_frcnn_neutral', 'SIMD_Conversion.m', 'SIMD_LGB', 'SIMD_LGB_test_nc15.yaml', 'SIMD_LGB_val_nc15.yaml', 'SIMD_RGB', 'SIMD_RGB_nc15.yaml', 'SIMD_RGB_test_nc15.yaml', 'SIMD_RGB_val_nc15.yaml', 'SIMD_YGB', 'SIMD_YGB_nc15.yaml', 'SIMD_YGB_test_nc15.yaml', 'SIMD_YGB_val_nc15.yaml', 'ssl_colorspace_comparison.m', 'ssl_colorspace_comparisonnew.m', 'SSL_Conversion.m', 'SSL_LGB', 'SSL_LGB_test_nc4.yaml', 'SSL_LGB_val_nc4.yaml', 'SSL_RGB', 'SSL_RGB_test_nc4.yaml', 'SSL_RGB_val_nc4.yaml', 'SSL_test.m', 'SSL_train.m', 'SSL_val.m', 'SSL_YGB', 'SSL

In [5]:
from pathlib import Path

BASE_DIR = Path(r"C:\Users\DELL\Desktop\c")        # folder that contains SIMD_RGB, SIMD_YGB, SIMD_LGB

# ============ the ONLY place you change the dataset ============
DATASET_NAME = "SIMD"                                  # "SSL" (4 classes)  or  "SIMD" (15 classes)
DATASET_CLASSES = {
    "SSL":  ['airplane', 'boat', 'car', 'ship'],       # order = class ids 0,1,2,3 in the YOLO label files (check against the yaml, see section below)
    "SIMD": ['car', 'truck', 'van', 'longvehicle', 'bus', 'airliner', 'propeller', 'trainer',
             'chartered', 'fighter', 'other', 'stairtruck', 'pushbacktruck', 'helicopter', 'boat'],
}
# ================================================================

DATASETS = {c: BASE_DIR / f"{DATASET_NAME}_{c}" for c in ("RGB", "YGB", "LGB")}
print("dataset:", DATASET_NAME)
print(DATASETS)

dataset: SIMD
{'RGB': WindowsPath('C:/Users/DELL/Desktop/c/SIMD_RGB'), 'YGB': WindowsPath('C:/Users/DELL/Desktop/c/SIMD_YGB'), 'LGB': WindowsPath('C:/Users/DELL/Desktop/c/SIMD_LGB')}


In [6]:
# (merged into the previous cell: BASE_DIR / DATASETS are defined once, above)

In [7]:

RUN_COLORS = ["RGB", "LGB", "YGB"]                       # which ones to train in this session
RUNS_DIR   = BASE_DIR / "runs_frcnn_neutral"      # NEW folder: the original runs are not touched

# ---- pretrained checkpoint
WEIGHTS_URL  = "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth"
WEIGHTS_FILE = "fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth"
WEIGHTS_DIR  = BASE_DIR / "pretrained"

# ---- classes (taken from DATASET_CLASSES above; same order as the YOLO data.yaml)
CLASS_NAMES = DATASET_CLASSES[DATASET_NAME]
NUM_CLASSES = len(CLASS_NAMES) + 1                       # +1 = background (label 0)

# ---- class imbalance
IMBALANCE_MODE = "ce"      # "none" | "rfs" (resample images) | "ce" (weighted loss) | "both"
RFS_T      = 0.1            # repeat-factor threshold: classes in < 10% of images get oversampled
CE_POWER   = 0.5            # 1.0 = full inverse frequency, 0.5 = square-root (softer, recommended)
CE_CLIP    = (0.1, 10.0)    # keep weights in a sane range

# ---- training
EPOCHS        = 40
BATCH_SIZE    = 2           # 2 fits the 6 GB RTX 2060; lower to 1 only if you get CUDA out-of-memory
LR            = 0.005
MOMENTUM      = 0.9
WEIGHT_DECAY  = 1e-4
WARMUP_ITERS  = 500
PATIENCE      = 10          # early stopping on validation mAP (YOLO notebook used 15)
GRAD_CLIP     = 10.0
USE_AMP       = True        # mixed precision: ~2x faster, much less VRAM
FREEZE_BN     = True
SEED          = 42
NUM_WORKERS   = 0 if os.name == "nt" else 4
MIN_SIZE, MAX_SIZE = 800, 1333   # the model resizes internally; keep resolution high because vehicles are small

RUNS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)

# ---- fair / colour-neutral start (NEW) ----------------------------------------------------------
NORM_MODE = "shared"       # "shared": one mean/std for ALL channels (recommended with COCO weights)
                           # "none"  : raw [0,1] input (mean 0, std 1). Not recommended: the pretrained layers expect
                           #           inputs ~4.4x larger in contrast (1/0.226), so everything starts mis-scaled.
if NORM_MODE == "shared":
    NORM_MEAN, NORM_STD = [0.449] * 3, [0.226] * 3     # channel-averaged ImageNet: mean(.485,.456,.406), std(.229,.224,.225)
elif NORM_MODE == "none":
    NORM_MEAN, NORM_STD = [0.0] * 3, [1.0] * 3
else:
    raise ValueError(NORM_MODE)
NEUTRAL_CONV1 = True       # average the pretrained conv1 filters over the 3 input channels
RECALIBRATE_BN = True      # re-estimate backbone BatchNorm statistics on this colour space's training images
RECAL_BATCHES  = 100       # 100 batches x BATCH_SIZE images
print("normalisation:", NORM_MODE, NORM_MEAN, NORM_STD, "| neutral conv1:", NEUTRAL_CONV1, "| BN recalibration:", RECALIBRATE_BN)


normalisation: shared [0.449, 0.449, 0.449] [0.226, 0.226, 0.226] | neutral conv1: True | BN recalibration: True


In [8]:
def rgb_to_ygb(img_rgb_uint8):
    # img: H x W x 3 uint8 (R,G,B)  ->  H x W x 3 uint8 (Y,G,B)
    x = img_rgb_uint8.astype(np.float32)
    Y = 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]
    out = x.copy()
    out[..., 0] = Y                     # replace the R channel
    return np.clip(out, 0, 255).round().astype(np.uint8)

pixel = np.array([[[200, 100, 50]]], dtype=np.uint8)
print("RGB pixel:", pixel[0, 0], "-> YGB pixel:", rgb_to_ygb(pixel)[0, 0])      # expect [124 100 50]

def iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union
print("IoU of two 10x10 boxes shifted by 5 px:", round(iou_xyxy([0,0,10,10], [5,0,15,10]), 3))   # expect 0.333

RGB pixel: [200 100  50] -> YGB pixel: [124 100  50]
IoU of two 10x10 boxes shifted by 5 px: 0.333


## Dataset: read the YOLO folders and give them to Faster R-CNN

In [9]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def find_split_dir(root, split):
    # case-insensitive: finds 'val' or 'VAL'
    for p in Path(root).iterdir():
        if p.is_dir() and p.name.lower() == split.lower():
            return p
    raise FileNotFoundError(f"No '{split}' folder inside {root}")

def parse_yolo_file(path):
    # returns array [n, 5] = (class, cx, cy, w, h); empty array if no file / no objects
    rows = []
    if Path(path).exists():
        for line in Path(path).read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                rows.append([float(v) for v in parts[:5]])
    return np.array(rows, dtype=np.float32).reshape(-1, 5)

class DetDataset(Dataset):
    def __init__(self, root, split, train=False):
        split_dir = find_split_dir(root, split)
        self.img_dir, self.lbl_dir = split_dir / "images", split_dir / "labels"
        self.paths = sorted(p for p in self.img_dir.iterdir() if p.suffix.lower() in IMG_EXTS)
        self.train = train
        if len(self.paths) == 0:
            raise RuntimeError(f"No images found in {self.img_dir}")

    def __len__(self):
        return len(self.paths)

    def label_path(self, i):
        return self.lbl_dir / (self.paths[i].stem + ".txt")

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")      # for YGB/LGB folders the 3 stored channels are simply (Y,G,B)/(L,G,B)
        W, H = img.size
        a = parse_yolo_file(self.label_path(i))
        cls = a[:, 0].astype(np.int64) + 1                  # +1 because 0 = background
        cx, cy, bw, bh = a[:, 1]*W, a[:, 2]*H, a[:, 3]*W, a[:, 4]*H
        boxes = np.stack([cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2], axis=1).astype(np.float32)
        boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, W)
        boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, H)
        keep = ((boxes[:, 2] - boxes[:, 0]) >= 1) & ((boxes[:, 3] - boxes[:, 1]) >= 1)
        boxes, cls = boxes[keep], cls[keep]

        img_t   = torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 255.0
        boxes_t = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels_t = torch.as_tensor(cls, dtype=torch.int64)

        if self.train:
            if random.random() < 0.5:                       # horizontal flip
                img_t = img_t.flip(-1)
                boxes_t[:, [0, 2]] = W - boxes_t[:, [2, 0]]
            if random.random() < 0.5:                       # vertical flip (valid for aerial images)
                img_t = img_t.flip(-2)
                boxes_t[:, [1, 3]] = H - boxes_t[:, [3, 1]]

        return img_t, {"boxes": boxes_t, "labels": labels_t, "image_id": torch.tensor([i])}

def collate_fn(batch):
    return tuple(zip(*batch))

# quick check
_ds = DetDataset(DATASETS["RGB"], "train", train=True)
_img, _tgt = _ds[0]
print("train images:", len(_ds), "| first image:", tuple(_img.shape), "| boxes:", tuple(_tgt["boxes"].shape), "| labels:", _tgt["labels"][:10].tolist())

train images: 4000 | first image: (3, 768, 1024) | boxes: (21, 4) | labels: [1, 1, 1, 1, 1, 3, 3, 1, 3, 3]


### Dataset sanity check (run once)
Checks that the 3 colour folders match, that class ids fit `CLASS_NAMES`, and shows the yaml so you can confirm the class **order** (id 0 = first name).

In [10]:
def dataset_sanity_check():
    print(f"Dataset: {DATASET_NAME} | {len(CLASS_NAMES)} classes: {CLASS_NAMES}\n")
    # 1) same image names in the three colour folders? (needed for a fair RGB/YGB/LGB comparison)
    for split in ("train", "val", "test"):
        names = {}
        for color, root in DATASETS.items():
            d = find_split_dir(root, split) / "images"
            names[color] = {p.stem for p in d.iterdir() if p.suffix.lower() in IMG_EXTS}
        same = all(names[c] == names["RGB"] for c in names)
        print(f"{split:5s} images: " + " | ".join(f"{c} {len(n)}" for c, n in names.items()) + f"  -> same names in all 3: {same}")
    # 2) objects per class actually present in the label files (class ids must be 0..N-1)
    print()
    for split in ("train", "val", "test"):
        lbl_dir = find_split_dir(DATASETS["RGB"], split) / "labels"
        ids, odd_lines = [], 0
        for f in lbl_dir.glob("*.txt"):
            for line in f.read_text().splitlines():
                p = line.split()
                if not p:
                    continue
                if len(p) != 5:
                    odd_lines += 1            # polygon / segmentation lines would show up here
                ids.append(int(float(p[0])))
        ids = np.array(ids, dtype=int)
        ok = (ids >= 0) & (ids < len(CLASS_NAMES))
        counts = np.bincount(ids[ok], minlength=len(CLASS_NAMES))
        print(f"{split:5s} objects per class: {dict(zip(CLASS_NAMES, counts.tolist()))}"
              f" | ids outside 0..{len(CLASS_NAMES)-1}: {int((~ok).sum())} | lines with != 5 values: {odd_lines}")
    # 3) print the yaml so you can compare its class order with CLASS_NAMES
    for y in sorted(BASE_DIR.glob(f"{DATASET_NAME}_RGB_*.yaml"))[:1]:
        print(f"\n--- {y.name} ---")
        print(y.read_text())

dataset_sanity_check()

Dataset: SIMD | 15 classes: ['car', 'truck', 'van', 'longvehicle', 'bus', 'airliner', 'propeller', 'trainer', 'chartered', 'fighter', 'other', 'stairtruck', 'pushbacktruck', 'helicopter', 'boat']

train images: RGB 4000 | YGB 4000 | LGB 4000  -> same names in all 3: True
val   images: RGB 500 | YGB 500 | LGB 500  -> same names in all 3: True
test  images: RGB 500 | YGB 500 | LGB 500  -> same names in all 3: True

train objects per class: {'car': 16241, 'truck': 2257, 'van': 4709, 'longvehicle': 1352, 'bus': 1580, 'airliner': 761, 'propeller': 162, 'trainer': 498, 'chartered': 499, 'fighter': 41, 'other': 654, 'stairtruck': 341, 'pushbacktruck': 150, 'helicopter': 47, 'boat': 6285} | ids outside 0..14: 0 | lines with != 5 values: 0
val   objects per class: {'car': 2080, 'truck': 309, 'van': 503, 'longvehicle': 158, 'bus': 226, 'airliner': 111, 'propeller': 17, 'trainer': 69, 'chartered': 65, 'fighter': 3, 'other': 70, 'stairtruck': 68, 'pushbacktruck': 37, 'helicopter': 9, 'boat': 1373}

## Class imbalance

In [11]:
def class_statistics(ds):
    # reads only label files (fast). Returns: instances per class, images per class, set-of-classes per image
    inst = np.zeros(len(CLASS_NAMES), dtype=np.float64)
    img_cnt = np.zeros(len(CLASS_NAMES), dtype=np.float64)
    per_img = []
    for i in range(len(ds)):
        ids = parse_yolo_file(ds.label_path(i))[:, 0].astype(int)
        ids = ids[(ids >= 0) & (ids < len(CLASS_NAMES))]
        per_img.append(set(ids.tolist()))
        inst += np.bincount(ids, minlength=len(CLASS_NAMES))
        for c in set(ids.tolist()):
            img_cnt[c] += 1
    return inst, img_cnt, per_img

def make_ce_weights(inst, power=0.5, clip=(0.1, 10.0)):
    n = np.maximum(inst, 1.0)
    w = n ** (-power)                       # rare class -> larger weight
    w = w * (n.sum() / (n * w).sum())       # frequency-weighted mean = 1  (same scale as background = 1)
    w = np.clip(w, *clip)
    w = w * (n.sum() / (n * w).sum())
    return torch.tensor(np.concatenate([[1.0], w]), dtype=torch.float32)   # index 0 = background

def make_rfs_weights(per_img, img_cnt, t=0.1):
    f = img_cnt / max(len(per_img), 1)
    r = np.maximum(1.0, np.sqrt(t / np.maximum(f, 1e-12)))                 # repeat factor per class
    w = np.array([max((r[c] for c in s), default=1.0) for s in per_img])   # image weight = max over its classes
    return r, w

def print_class_table(inst, img_cnt, r=None, ce_w=None):
    df = pd.DataFrame({"class": CLASS_NAMES, "instances": inst.astype(int), "images": img_cnt.astype(int)})
    if r is not None:    df["RFS_repeat_factor"] = np.round(r, 2)
    if ce_w is not None: df["CE_weight"] = np.round(ce_w.numpy()[1:], 3)
    print(df.to_string(index=False))

# ---- weighted loss that replaces torchvision's fastrcnn_loss
_ORIG_FASTRCNN_LOSS = tv_roi_heads.fastrcnn_loss
_announced = {"done": False}

def install_weighted_loss(class_weights):
    cw = class_weights.clone()
    def weighted_fastrcnn_loss(class_logits, box_regression, labels, regression_targets):
        labels = torch.cat(labels, dim=0)
        regression_targets = torch.cat(regression_targets, dim=0)
        w = cw.to(class_logits.device)
        classification_loss = F.cross_entropy(class_logits.float(), labels, weight=w)   # <-- the only change
        pos = torch.where(labels > 0)[0]
        labels_pos = labels[pos]
        N = class_logits.shape[0]
        box_regression = box_regression.float().reshape(N, box_regression.size(-1) // 4, 4)
        box_loss = F.smooth_l1_loss(box_regression[pos, labels_pos], regression_targets[pos], beta=1/9, reduction="sum")
        box_loss = box_loss / labels.numel()
        if not _announced["done"]:
            wt = cw.cpu().numpy()
            names = ["background"] + list(CLASS_NAMES)
            lo, hi = 1 + int(np.argmin(wt[1:])), 1 + int(np.argmax(wt[1:]))
            print(">>> WEIGHTED classification loss is ACTIVE")
            print("    weights: " + " | ".join(f"{n}={v:.3f}" for n, v in zip(names, wt)))
            print(f"    lowest class weight: {names[lo]} ({wt[lo]:.3f}) | highest: {names[hi]} ({wt[hi]:.3f})")
            _announced["done"] = True
        return classification_loss, box_loss
    tv_roi_heads.fastrcnn_loss = weighted_fastrcnn_loss
    _announced["done"] = False

def restore_default_loss():
    tv_roi_heads.fastrcnn_loss = _ORIG_FASTRCNN_LOSS

##  Build the model and load the pretrained COCO weights

## Colour-neutral helpers (new)

In [12]:
@torch.no_grad()
def make_stem_channel_neutral(model):
    """conv1 weight is [64, 3, 7, 7]. Replace the 3 input-channel filters by their mean, repeated 3 times.
    Response to a grey patch (R=G=B) is unchanged (3 * mean * g = sum(W_c) * g); colour-opponent filters (R-G ...) are removed,
    and every input channel now has the SAME filter, so no colour space is favoured at the first layer."""
    w = model.backbone.body.conv1.weight                    # nn.Parameter, stays trainable
    w.copy_(w.mean(dim=1, keepdim=True).repeat(1, w.shape[1], 1, 1))

@torch.no_grad()
def recalibrate_backbone_bn(model, loader, n_batches=100):
    """Adaptive BN: exact average of the BN statistics over n_batches of THIS colour space (momentum=None -> cumulative mean)."""
    bns = [m for m in model.backbone.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
    saved = [m.momentum for m in bns]
    model.eval()                                            # transform/resize in eval mode, everything else frozen
    for m in bns:
        m.reset_running_stats(); m.momentum = None; m.train()   # only the backbone BN layers collect statistics
    for k, (images, _) in enumerate(loader):
        if k >= n_batches:
            break
        image_list, _ = model.transform([im.to(device) for im in images])    # normalise + resize exactly as in training
        model.backbone(image_list.tensors)
    for m, mo in zip(bns, saved):
        m.eval(); m.momentum = mo
    print(f"  BN recalibrated on {min(k + 1, n_batches)} batches ({len(bns)} BatchNorm layers in the backbone)")


In [13]:
def build_model():
    model = fasterrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None,
        min_size=MIN_SIZE, max_size=MAX_SIZE,
        box_score_thresh=0.001, box_detections_per_img=300,
        image_mean=NORM_MEAN, image_std=NORM_STD,                    # NEW: same constants for all three channels
    )
    state = torch.hub.load_state_dict_from_url(
        WEIGHTS_URL, model_dir=str(WEIGHTS_DIR), file_name=WEIGHTS_FILE,
        map_location="cpu", progress=True, check_hash=True, weights_only=True,
    )
    print("Loaded checkpoint:", WEIGHTS_DIR / WEIGHTS_FILE)
    model.load_state_dict(state)                                     # 91-class COCO model (head is replaced just below)
    if NEUTRAL_CONV1:
        make_stem_channel_neutral(model)                             # NEW: identical conv1 filters for channel 0, 1, 2
    in_features = model.roi_heads.box_predictor.cls_score.in_features   # 1024
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)   # new head: background + all classes
    assert list(model.transform.image_mean) == list(NORM_MEAN) and list(model.transform.image_std) == list(NORM_STD)
    return model

def set_bn_eval(model):
    for m in model.modules():
        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
            m.eval()

def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, trn


## Smoke test (run this once before the long training)

In [14]:
def smoke_test(color="RGB"):
    ds = DetDataset(DATASETS[color], "train", train=True)
    inst, img_cnt, per_img = class_statistics(ds)
    restore_default_loss()
    ce_w = None
    if IMBALANCE_MODE in ("ce", "both"):
        ce_w = make_ce_weights(inst, CE_POWER, CE_CLIP)
        install_weighted_loss(ce_w)
    print_class_table(inst, img_cnt, None, ce_w)
    model = build_model().to(device)
    w = model.backbone.body.conv1.weight
    print("normalisation:", model.transform.image_mean, model.transform.image_std)
    print("stem filters identical across input channels:", bool(torch.allclose(w[:, 0], w[:, 1]) and torch.allclose(w[:, 0], w[:, 2])), "| expected:", NEUTRAL_CONV1)
    if RECALIBRATE_BN:
        recalibrate_backbone_bn(model, DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn), n_batches=3)
    print("params total/trainable: %.1fM / %.1fM" % tuple(x/1e6 for x in count_params(model)))
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    images, targets = next(iter(loader))
    images  = [im.to(device) for im in images]
    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
    model.train(); set_bn_eval(model)
    with torch.autocast(device_type="cuda", enabled=USE_AMP and device.type == "cuda"):
        losses = model(images, targets)
    print({k: round(v.item(), 4) for k, v in losses.items()})
    del model; torch.cuda.empty_cache()

smoke_test("RGB")

        class  instances  images  CE_weight
          car      16241    2250      0.504
        truck       2257     796      1.352
          van       4709    1806      0.936
  longvehicle       1352     421      1.747
          bus       1580     513      1.616
     airliner        761     618      2.328
    propeller        162     118      5.047
      trainer        498     160      2.878
    chartered        499     282      2.876
      fighter         41      26     10.000
        other        654     318      2.512
   stairtruck        341     206      3.478
pushbacktruck        150     101      5.245
   helicopter         47      38      9.370
         boat       6285     337      0.810
Loaded checkpoint: C:\Users\DELL\Desktop\c\pretrained\fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
normalisation: [0.449, 0.449, 0.449] [0.226, 0.226, 0.226]
stem filters identical across input channels: True | expected: True
  BN recalibrated on 3 batches (61 BatchNorm layers in the backbone)
p

## Training and evaluation functions

In [15]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, log_every=50):
    model.train()
    if FREEZE_BN:
        set_bn_eval(model)
    totals, n_ok = {}, 0
    t0 = time.time()
    for it, (images, targets) in enumerate(loader, 1):
        images  = [im.to(device, non_blocking=True) for im in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        with torch.autocast(device_type="cuda", enabled=USE_AMP and device.type == "cuda"):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
        if not torch.isfinite(loss):
            print("  non-finite loss, skipping batch")
            optimizer.zero_grad(set_to_none=True)
            continue
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        for k, v in loss_dict.items():
            totals[k] = totals.get(k, 0.0) + v.item()
        n_ok += 1
        if it % log_every == 0:
            el = time.time() - t0
            eta = el / it * (len(loader) - it)
            print(f"  iter {it}/{len(loader)} | loss {loss.item():.3f} | "
                  f"{it/el:.2f} it/s | epoch ETA {eta/60:.1f} min", flush=True)
    return {k: v / max(n_ok, 1) for k, v in totals.items()}

@torch.no_grad()
def evaluate(model, loader, per_class=False):
    model.eval()
    m_all = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    m_50  = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", iou_thresholds=[0.5], class_metrics=per_class)
    for images, targets in loader:
        images = [im.to(device) for im in images]
        outputs = model(images)
        preds = [{k: v.detach().cpu() for k, v in o.items()} for o in outputs]
        gts   = [{"boxes": t["boxes"], "labels": t["labels"]} for t in targets]
        m_all.update(preds, gts)
        m_50.update(preds, gts)
    r, r50 = m_all.compute(), m_50.compute()
    res = {"mAP50-95": r["map"].item(), "mAP50": r["map_50"].item(), "mAP75": r["map_75"].item(), "mAR100": r["mar_100"].item()}
    if per_class:
        classes = r50["classes"].tolist()
        aps = r50["map_per_class"].tolist()
        res["per_class_AP50"] = {CLASS_NAMES[c - 1]: round(a, 4) for c, a in zip(classes, aps)}   # label 1..N -> name
    return res

## One full experiment for one colour space

In [16]:
def run_experiment(color):
    seed_everything(SEED)
    root = DATASETS[color]
    run_dir = RUNS_DIR / f"{DATASET_NAME}_{color}_FasterRCNN_{IMBALANCE_MODE}"
    run_dir.mkdir(parents=True, exist_ok=True)
    print("=" * 70, f"\n{color} | imbalance mode: {IMBALANCE_MODE} | output: {run_dir}\n" + "=" * 70)

    train_ds = DetDataset(root, "train", train=True)
    val_ds   = DetDataset(root, "val",   train=False)
    test_ds  = DetDataset(root, "test",  train=False)
    print(f"images  train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

    # ---- class imbalance handling
    inst, img_cnt, per_img = class_statistics(train_ds)
    sampler, r, ce_w = None, None, None
    if IMBALANCE_MODE in ("rfs", "both"):
        r, img_w = make_rfs_weights(per_img, img_cnt, RFS_T)
        sampler = WeightedRandomSampler(torch.as_tensor(img_w, dtype=torch.double), num_samples=len(train_ds),
                                        replacement=True, generator=torch.Generator().manual_seed(SEED))
    restore_default_loss()
    if IMBALANCE_MODE in ("ce", "both"):
        ce_w = make_ce_weights(inst, CE_POWER, CE_CLIP)
        install_weighted_loss(ce_w)
    print_class_table(inst, img_cnt, r, ce_w)

    # ---- loaders
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=(device.type == "cuda"))
    train_loader = DataLoader(train_ds, sampler=sampler, shuffle=(sampler is None), drop_last=True, **kw)
    val_loader   = DataLoader(val_ds,  shuffle=False, **kw)
    test_loader  = DataLoader(test_ds, shuffle=False, **kw)

    # ---- model / optimiser / scheduler
    model = build_model().to(device)
    if RECALIBRATE_BN:
        recalibrate_backbone_bn(model, train_loader, RECAL_BATCHES)   # NEW: BN statistics of THIS colour space
        seed_everything(SEED)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    total_iters = EPOCHS * len(train_loader)
    def lr_lambda(it):
        if it < WARMUP_ITERS:
            return 0.001 + 0.999 * it / WARMUP_ITERS
        p = (it - WARMUP_ITERS) / max(1, total_iters - WARMUP_ITERS)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * p))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and device.type == "cuda")

    # ---- training loop with early stopping
    best, bad, history = -1.0, 0, []
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
        va = evaluate(model, val_loader)
        row = {"epoch": epoch, **{k: round(v, 4) for k, v in tr.items()}, **{f"val_{k}": round(v, 4) for k, v in va.items()},
               "lr": optimizer.param_groups[0]["lr"], "min": round((time.time() - t0) / 60, 2)}
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        print(f"ep {epoch:02d} | cls {tr.get('loss_classifier', 0):.3f} box {tr.get('loss_box_reg', 0):.3f} "
              f"obj {tr.get('loss_objectness', 0):.3f} rpn {tr.get('loss_rpn_box_reg', 0):.3f} | "
              f"val mAP50-95 {va['mAP50-95']:.4f} mAP50 {va['mAP50']:.4f} | {row['min']} min")
        if va["mAP50-95"] > best:
            best, bad = va["mAP50-95"], 0
            torch.save(model.state_dict(), run_dir / "best.pth")
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"early stopping (no val improvement for {PATIENCE} epochs)")
                break

    # ---- final evaluation with the best checkpoint
    model.load_state_dict(torch.load(run_dir / "best.pth", map_location=device, weights_only=True))
    val_res  = evaluate(model, val_loader,  per_class=True)
    test_res = evaluate(model, test_loader, per_class=True)
    out = {"dataset": DATASET_NAME, "color": color, "mode": IMBALANCE_MODE, "norm_mode": NORM_MODE, "norm_mean": NORM_MEAN, "norm_std": NORM_STD, "neutral_conv1": NEUTRAL_CONV1, "recalibrate_bn": RECALIBRATE_BN, "best_val_mAP50-95": best, "val": val_res, "test": test_res}
    (run_dir / "results.json").write_text(json.dumps(out, indent=2))
    print("\nTEST:", {k: round(v, 4) for k, v in test_res.items() if k != "per_class_AP50"})
    for name, ap in test_res["per_class_AP50"].items():
        print(f"   {name:15s} AP50 {ap:.4f}")
    del model, optimizer; torch.cuda.empty_cache()
    return out

## experiment

all of them are training- [rgb,ygb and lgb all] takes time keep it like this

In [17]:
all_results = {}
for c in RUN_COLORS:
    all_results[c] = run_experiment(c)

RGB | imbalance mode: ce | output: C:\Users\DELL\Desktop\c\runs_frcnn_neutral\SIMD_RGB_FasterRCNN_ce
images  train=4000  val=500  test=500
        class  instances  images  CE_weight
          car      16241    2250      0.504
        truck       2257     796      1.352
          van       4709    1806      0.936
  longvehicle       1352     421      1.747
          bus       1580     513      1.616
     airliner        761     618      2.328
    propeller        162     118      5.047
      trainer        498     160      2.878
    chartered        499     282      2.876
      fighter         41      26     10.000
        other        654     318      2.512
   stairtruck        341     206      3.478
pushbacktruck        150     101      5.245
   helicopter         47      38      9.370
         boat       6285     337      0.810
Loaded checkpoint: C:\Users\DELL\Desktop\c\pretrained\fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
  BN recalibrated on 100 batches (61 BatchNorm layers in t

C:\Users\DELL\miniconda3\Lib\site-packages\torch\optim\lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


  iter 50/2000 | loss 1.090 | 2.93 it/s | epoch ETA 11.1 min
  iter 100/2000 | loss 1.607 | 2.93 it/s | epoch ETA 10.8 min
  iter 150/2000 | loss 0.995 | 2.93 it/s | epoch ETA 10.5 min
  iter 200/2000 | loss 0.712 | 2.93 it/s | epoch ETA 10.2 min
  iter 250/2000 | loss 0.568 | 2.93 it/s | epoch ETA 10.0 min
  iter 300/2000 | loss 0.854 | 2.93 it/s | epoch ETA 9.7 min
  iter 350/2000 | loss 0.869 | 2.93 it/s | epoch ETA 9.4 min
  iter 400/2000 | loss 0.943 | 2.93 it/s | epoch ETA 9.1 min
  iter 450/2000 | loss 0.374 | 2.93 it/s | epoch ETA 8.8 min
  iter 500/2000 | loss 0.589 | 2.93 it/s | epoch ETA 8.5 min
  iter 550/2000 | loss 0.888 | 2.92 it/s | epoch ETA 8.3 min
  iter 600/2000 | loss 0.679 | 2.92 it/s | epoch ETA 8.0 min
  iter 650/2000 | loss 0.681 | 2.92 it/s | epoch ETA 7.7 min
  iter 700/2000 | loss 0.985 | 2.91 it/s | epoch ETA 7.4 min
  iter 750/2000 | loss 0.890 | 2.91 it/s | epoch ETA 7.2 min
  iter 800/2000 | loss 0.809 | 2.91 it/s | epoch ETA 6.9 min
  iter 850/2000 | lo

C:\Users\DELL\miniconda3\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


ep 01 | cls 0.485 box 0.237 obj 0.022 rpn 0.013 | val mAP50-95 0.3048 mAP50 0.4922 | 12.87 min
  iter 50/2000 | loss 0.647 | 2.82 it/s | epoch ETA 11.5 min
  iter 100/2000 | loss 0.291 | 2.83 it/s | epoch ETA 11.2 min
  iter 150/2000 | loss 0.288 | 2.82 it/s | epoch ETA 10.9 min
  iter 200/2000 | loss 0.356 | 2.82 it/s | epoch ETA 10.6 min
  iter 250/2000 | loss 0.669 | 2.82 it/s | epoch ETA 10.3 min
  iter 300/2000 | loss 0.296 | 2.82 it/s | epoch ETA 10.0 min
  iter 350/2000 | loss 0.535 | 2.82 it/s | epoch ETA 9.8 min
  iter 400/2000 | loss 0.528 | 2.82 it/s | epoch ETA 9.5 min
  iter 450/2000 | loss 0.388 | 2.82 it/s | epoch ETA 9.2 min
  iter 500/2000 | loss 0.724 | 2.82 it/s | epoch ETA 8.9 min
  iter 550/2000 | loss 0.687 | 2.82 it/s | epoch ETA 8.6 min
  iter 600/2000 | loss 0.506 | 2.82 it/s | epoch ETA 8.3 min
  iter 650/2000 | loss 0.179 | 2.82 it/s | epoch ETA 8.0 min
  iter 700/2000 | loss 0.355 | 2.82 it/s | epoch ETA 7.7 min
  iter 750/2000 | loss 0.284 | 2.82 it/s | epo

C:\Users\DELL\miniconda3\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


... (223 outputs left out)

C:\Users\DELL\miniconda3\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


ep 40 | cls 0.049 box 0.089 obj 0.001 rpn 0.004 | val mAP50-95 0.6699 mAP50 0.8362 | 13.03 min


C:\Users\DELL\miniconda3\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)



TEST: {'mAP50-95': 0.6623, 'mAP50': 0.8397, 'mAP75': 0.7806, 'mAR100': 0.7847}
   car             AP50 0.9355
   truck           AP50 0.8085
   van             AP50 0.8713
   longvehicle     AP50 0.8491
   bus             AP50 0.8979
   airliner        AP50 0.9849
   propeller       AP50 0.9959
   trainer         AP50 0.9899
   chartered       AP50 0.9649
   fighter         AP50 1.0000
   other           AP50 0.3187
   stairtruck      AP50 0.4730
   pushbacktruck   AP50 0.5343
   helicopter      AP50 1.0000
   boat            AP50 0.9713


## 12. Compare RGB vs YGB vs LGB

In [18]:
rows, per_cls = [], {}
print("dataset:", DATASET_NAME)
for c in ["RGB", "YGB", "LGB"]:
    f = RUNS_DIR / f"{DATASET_NAME}_{c}_FasterRCNN_{IMBALANCE_MODE}" / "results.json"
    if f.exists():
        d = json.loads(f.read_text())
        rows.append({"color": c, **{k: round(v, 4) for k, v in d["test"].items() if k != "per_class_AP50"}})
        per_cls[c] = d["test"]["per_class_AP50"]
print(pd.DataFrame(rows).to_string(index=False))
print()
print(pd.DataFrame(per_cls).to_string())

dataset: SIMD
color  mAP50-95  mAP50  mAP75  mAR100
  RGB    0.6558 0.8499 0.7778  0.7872
  YGB    0.6623 0.8397 0.7806  0.7847
  LGB    0.6577 0.8401 0.7825  0.7940

                  RGB     YGB     LGB
car            0.9375  0.9355  0.9340
truck          0.8121  0.8085  0.7996
van            0.8894  0.8713  0.8879
longvehicle    0.8676  0.8491  0.8370
bus            0.9159  0.8979  0.9125
airliner       0.9716  0.9849  0.9679
propeller      0.9959  0.9959  1.0000
trainer        0.9938  0.9899  0.9934
chartered      0.9529  0.9649  0.9598
fighter        1.0000  1.0000  1.0000
other          0.3212  0.3187  0.3591
stairtruck     0.5076  0.4730  0.4720
pushbacktruck  0.6059  0.5343  0.5028
helicopter     1.0000  1.0000  1.0000
boat           0.9778  0.9713  0.9761


In [19]:
RUN_PREFIX = "SIMD"

In [20]:
print("RUN_PREFIX:", RUN_PREFIX)
for c in ["RGB","YGB","LGB"]:
    p = RUNS_DIR / f"{RUN_PREFIX}_{c}_FasterRCNN_{'ce'}" / "best.pth"
    print(c, "->", p.exists(), p)

RUN_PREFIX: SIMD
RGB -> True C:\Users\DELL\Desktop\c\runs_frcnn_neutral\SIMD_RGB_FasterRCNN_ce\best.pth
YGB -> True C:\Users\DELL\Desktop\c\runs_frcnn_neutral\SIMD_YGB_FasterRCNN_ce\best.pth
LGB -> True C:\Users\DELL\Desktop\c\runs_frcnn_neutral\SIMD_LGB_FasterRCNN_ce\best.pth


In [21]:

from torchvision.ops import box_iou
from tqdm.auto import tqdm

CONF_THR   = 0.5          # score threshold used ONLY for precision / recall
IOU_THR    = 0.5          # IoU needed to count a detection as correct
RUN_PREFIX = globals().get("DATASET_NAME", "SSL")          # SSL notebook -> "SSL", SIMD notebook -> "SIMD"
DS         = globals().get("DetDataset") or globals().get("SIMDDataset")
NC         = len(CLASS_NAMES)

def match_counts(pred, gt):
    """Greedy matching (highest score first). Returns per-class TP, FP, FN arrays (index 0..NC-1)."""
    tp = np.zeros(NC, dtype=np.int64); fp = np.zeros(NC, dtype=np.int64); fn = np.zeros(NC, dtype=np.int64)
    keep = pred["scores"] >= CONF_THR
    pb, ps, pl = pred["boxes"][keep], pred["scores"][keep], pred["labels"][keep]
    order = ps.argsort(descending=True)
    pb, pl = pb[order], pl[order]
    gb, gl = gt["boxes"], gt["labels"]
    used = torch.zeros(len(gb), dtype=torch.bool)
    ious = box_iou(pb, gb) if len(pb) and len(gb) else torch.zeros(len(pb), len(gb))
    for i in range(len(pb)):
        c = int(pl[i]) - 1                                   # label 1..N -> index 0..N-1
        cand = ious[i].clone()
        if len(cand) > 0:
            cand[(gl != pl[i]) | used] = -1                  # wrong class or already matched
        if len(cand) > 0 and cand.max() >= IOU_THR:
            used[cand.argmax()] = True
            tp[c] += 1
        else:
            fp[c] += 1
    for g in gl[~used]:                                      # GT boxes nobody matched = missed objects
        fn[int(g) - 1] += 1
    return tp, fp, fn

@torch.no_grad()
def test_one(color):
    ckpt = RUNS_DIR / f"{RUN_PREFIX}_{color}_FasterRCNN_{IMBALANCE_MODE}" / "best.pth"
    if not ckpt.exists():
        print(f"[{color}] skipped, not found: {ckpt}")
        return None
    model = build_model()
    model.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=True))
    model.to(device).eval()
    loader = DataLoader(DS(DATASETS[color], "test", train=False), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
    m_all = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    m_50  = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", iou_thresholds=[0.5])
    TP = np.zeros(NC, dtype=np.int64); FP = TP.copy(); FN = TP.copy()
    for images, targets in tqdm(loader, desc=f"test {color}"):
        outs  = model([im.to(device) for im in images])
        preds = [{k: v.detach().cpu() for k, v in o.items()} for o in outs]
        gts   = [{"boxes": t["boxes"], "labels": t["labels"]} for t in targets]
        m_all.update(preds, gts); m_50.update(preds, gts)
        for p, g in zip(preds, gts):
            tp, fp, fn = match_counts(p, g)
            TP += tp; FP += fp; FN += fn
    del model; torch.cuda.empty_cache()

    P_c = TP / np.maximum(TP + FP, 1)                         # per-class precision
    R_c = TP / np.maximum(TP + FN, 1)                         # per-class recall
    P, R = TP.sum() / max(TP.sum() + FP.sum(), 1), TP.sum() / max(TP.sum() + FN.sum(), 1)   # overall (micro)
    return {
        "summary": {"color": color,
                    "mAP50-95":  round(m_all.compute()["map"].item(), 4),
                    "mAP50":     round(m_50.compute()["map_50"].item(), 4),
                    "Precision": round(float(P), 4),
                    "Recall":    round(float(R), 4),
                    "F1":        round(float(2 * P * R / max(P + R, 1e-9)), 4),
                    "mP(macro)": round(float(P_c.mean()), 4),
                    "mR(macro)": round(float(R_c.mean()), 4),
                    "TP": int(TP.sum()), "FP": int(FP.sum()), "FN": int(FN.sum())},
        "per_class": pd.DataFrame({"class": CLASS_NAMES, "TP": TP, "FP": FP, "FN": FN,
                                   "Precision": P_c.round(4), "Recall": R_c.round(4)}),
    }

results = {c: r for c in ["RGB", "YGB", "LGB"] if (r := test_one(c))}

test_df = pd.DataFrame([r["summary"] for r in results.values()])
print(f"\nTEST set | {RUN_PREFIX} | Precision/Recall at score >= {CONF_THR}, IoU >= {IOU_THR}")
print(test_df.to_string(index=False))

for c, r in results.items():
    print(f"\n--- {c}: per-class ---")
    print(r["per_class"].to_string(index=False))

out_csv = RUNS_DIR / f"{RUN_PREFIX}_test_precision_recall_conf{CONF_THR}.csv"
test_df.to_csv(out_csv, index=False)
print("\nsaved:", out_csv)

Loaded checkpoint: C:\Users\DELL\Desktop\c\pretrained\fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


test RGB:   0%|          | 0/250 [00:00<?, ?it/s]

C:\Users\DELL\miniconda3\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


Loaded checkpoint: C:\Users\DELL\Desktop\c\pretrained\fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


test YGB:   0%|          | 0/250 [00:00<?, ?it/s]

Loaded checkpoint: C:\Users\DELL\Desktop\c\pretrained\fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


test LGB:   0%|          | 0/250 [00:00<?, ?it/s]


TEST set | SIMD | Precision/Recall at score >= 0.5, IoU >= 0.5
color  mAP50-95  mAP50  Precision  Recall     F1  mP(macro)  mR(macro)   TP   FP  FN
  RGB    0.6558 0.8499     0.7805  0.9330 0.8500     0.7465     0.8936 4318 1214 310
  YGB    0.6623 0.8397     0.8091  0.9196 0.8608     0.7554     0.8599 4256 1004 372
  LGB    0.6577 0.8401     0.7768  0.9285 0.8459     0.7232     0.8812 4297 1235 331

--- RGB: per-class ---
        class   TP  FP  FN  Precision  Recall
          car 2056 550 127     0.7889  0.9418
        truck  195  89  41     0.6866  0.8263
          van  472 225  48     0.6772  0.9077
  longvehicle  101  72  11     0.5838  0.9018
          bus  174  55  11     0.7598  0.9405
     airliner   91   8   5     0.9192  0.9479
    propeller   16   1   0     0.9412  1.0000
      trainer   63   5   1     0.9265  0.9844
    chartered   76  18   0     0.8085  1.0000
      fighter    5   0   0     1.0000  1.0000
        other   36  75  25     0.3243  0.5902
   stairtruck   24  